# 第四週：Python × 語音 AI × 多語字幕影片
180 分鐘｜前三週語法整合｜CPU 可執行 Colab 教材

來源：GitHub 五週 Python 人工智慧教學包／第4週；Hugging Face Multilingual Video Studio V2 為延伸展示。先完成 JSON/SRT，不把這項練習視為已完成語音合成或翻譯。

## 第 1 單元（0–30 分）：import、變數與輸出資料夾
**教學 10 分 → 跟做 15 分 → 驗收 5 分。**

In [ ]:
from pathlib import Path
import json
course = "多語字幕影片工作坊"
output_dir = Path("week4_output")
output_dir.mkdir(exist_ok=True)
print(course, output_dir.resolve())

**逐行說明：**`import` 匯入；`=` 指派；`Path` 操作路徑；`mkdir` 建立目錄；`print` 顯示結果。改一行：將 `course` 更改為自己的作品名稱。

## 第 2 單元（30–60 分）：List／Dictionary／for
用三個場景描述圖片、旁白和每段預定長度。英文是預先撰寫的資料，**不是自動翻譯**。

In [ ]:
scenes = [
    {"id": 1, "image": "slide01.png", "zh": "歡迎來到多語影片實作。", "en": "Welcome to the video lab.", "duration": 3.0},
    {"id": 2, "image": "slide02.png", "zh": "使用 Python 管理多個場景。", "en": "Python organizes scenes.", "duration": 4.0},
    {"id": 3, "image": "slide03.png", "zh": "最後產出字幕和紀錄。", "en": "Finally export subtitles.", "duration": 3.5},
]
lang = "zh"
for scene in scenes:
    print(scene["id"], scene[lang])
assert len(scenes) == 3

**逐行說明：**`[...]` 是串列；`{...}` 是字典；`scene[lang]` 用鍵取值；`for` 逐一讀出。**練習：**改 `lang = "en"` 再執行。

## 第 3 單元（60–90 分）：def、if、raise、try / except
逐行認識輸入契約與錯誤處理。

In [ ]:
def validate_scene(item, language="zh"):
    for field in ("id", "image", language, "duration"):
        if field not in item:
            raise ValueError(f"缺少欄位: {field}")
    if not isinstance(item[language], str) or not item[language].strip():
        raise ValueError("旁白不可空白")
    if not isinstance(item["duration"], (int, float)) or not 0 < item["duration"] <= 60:
        raise ValueError("時間必須為 0 到 60 秒")
    return True
for scene in scenes:
    assert validate_scene(scene, lang)
try:
    validate_scene({"id": 4, "image": "no.png", "zh": "", "duration": 2})
except ValueError as error:
    print("錯誤防呆成功:", error)

**逐行說明：**`def` 宣告函式；`if` 判斷；`raise` 主動拒絕錯誤資料；`return` 回傳；`try/except` 捕捉例外。**驗收：**空白旁白會被拒絕。

## 第 4 單元（90–120 分）：時間軸、字串格式化與 SRT
以每段預定時長累加時間，不代表完成語音逐字對齊。

In [ ]:
def stamp(seconds):
    ms = round(seconds * 1000)
    hour, ms = divmod(ms, 3600000)
    minute, ms = divmod(ms, 60000)
    sec, milli = divmod(ms, 1000)
    return f"{hour:02}:{minute:02}:{sec:02},{milli:03}"

def build_cues(items, language):
    cursor = 0.0
    result = []
    for item in items:
        validate_scene(item, language)
        start = cursor
        cursor += item["duration"]
        result.append({"start": start, "end": cursor, "text": item[language]})
    return result

cues = build_cues(scenes, lang)
blocks = []
for num, cue in enumerate(cues, 1):
    blocks.append(f'{num}\n{stamp(cue["start"])} --> {stamp(cue["end"])}\n{cue["text"]}')
srt_text = "\n\n".join(blocks) + "\n"
(output_dir / "subtitles.srt").write_text(srt_text, encoding="utf-8-sig")
(output_dir / "scene_plan.json").write_text(json.dumps(scenes, ensure_ascii=False, indent=2), encoding="utf-8")
print(srt_text)

**逐行說明：**`divmod` 同時求商與餘數；`f-string` 格式化時間；`enumerate(..., 1)` 產生字幕序號；`write_text` 寫入檔案。**實作：**修改 duration 並觀察後續段落起點。

## 第 5 單元（120–150 分）：Colab 與 GitHub V2 工具對接
GitHub 的 `subtitle_tools.py`、`cues.example.json`、`test_subtitle_tools.py` 有自己的資料契約。不要直接假設本 Notebook 的 `scenes` 格式等於該工具輸入。

In [ ]:
for name in ("scene_plan.json", "subtitles.srt"):
    path = output_dir / name
    print(name, path.exists(), path.stat().st_size if path.exists() else 0)

**教師補充：**原有字幕工具在 FFmpeg 支援下可做 soft/burn；本程式只完成 JSON/SRT，沒有執行 TTS、ASR、翻譯或 MP4 生成。

## 第 6 單元（150–180 分）：成果驗收、Hugging Face UI 展示
參考：https://huggingface.co/spaces/wangsongwen/Multilingual-Video-Studio-V2 。若服務未啟用，使用本機成果完成課堂驗收。

In [ ]:
report = {"scene_count": len(scenes), "language": lang, "total_seconds": cues[-1]["end"], "tts_executed": False, "translation_executed": False, "video_rendered": False}
(output_dir / "run_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
assert len(scenes) >= 3
assert " --> " in srt_text
assert all(cues[i]["end"] <= cues[i+1]["start"] for i in range(len(cues)-1))
print("驗收通過：", report)

## 交付及評分
語法說明 30 分；資料驗證 25 分；SRT 正確性 25 分；執行紀錄與限制說明 20 分。

繳交：`scene_plan.json`、`subtitles.srt`、`run_report.json`、一段學習反思。下一週可延伸 CSV、Gradio 和影片處理。